# Milestone 1 Notebook
This jupyter notebook contains the exploratory part of the milestone 1, along with some preprocessing steps which are briefly described in the report.

In [1]:
import json
import pandas as pd
import nltk
import string
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
import stanza
from collections import Counter
import re

c:\Users\maria\anaconda3\envs\tuwnlpie\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Download stopwords from nltk
nltk.download("punkt")
nltk.download("stopwords")
stanza.download("en")

# Download lemmatizer and other requirements from nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

In [ ]:
def extract_statistics(filepath):
    """
    This function takes as input the path to a json file and returns some statistics.
    """
    with open(filepath, "r") as file:
        data = json.load(file)
    document_token_counts = []

    for entry in data:
        document_id = entry["docid"]
        token_count = len(entry["token"])
        document_token_counts.append({"docid": document_id, "token_count": token_count})

    # Create a DataFrame to store the document token counts
    document_token_counts_df = pd.DataFrame(document_token_counts)

    # Print the DataFrame statistics
    print(f"Number of sentences {len(document_token_counts_df)}")
    print(
        f"Average length of sentence {round(document_token_counts_df['token_count'].mean(), 1)}"
    )
    print(f"Min length of sentence {document_token_counts_df['token_count'].min()}")
    print(f"Max length of sentence {document_token_counts_df['token_count'].max()}")

    relations = []
    for entry in data:
        relations = relations + [entry["relation"]]

    cnt = Counter(relations)
    print(
        f"Percentage of no_relation entries {round(cnt['no_relation']/len(relations)*100, 1)}%"
    )

## Extract data statistics

In [ ]:
print("--- TACRED ---")
print("Train data")
extract_statistics("../../data/tacred/json/train.json")

print("\nDev data")
extract_statistics("../../data/tacred/json/dev.json")

print("\nTest data")
extract_statistics("../../data/tacred/json/test.json")


print("--- TACREV ---")
print("\nDev data")
extract_statistics("../../data/tacrev/json/dev.json")

print("\nTest data")
extract_statistics("../../data/tacrev/json/test.json")

In [ ]:
print("--- TACREV ---")
print("\nDev data")
extract_statistics("../../data/tacrev/json/dev.json")

print("\nTest data")
extract_statistics("../../data/tacrev/json/test.json")

All explorations from now on are performed using the training dataset.

In [2]:
# Now we work with the json file and open the training dataset.
with open("../../data/tacred/json/train.json", "r") as file:
    data = json.load(file)

In [ ]:
# Now we can count and visualise different subject types and object types
subject_types = [None] * len(data)
object_types = [None] * len(data)

i = 0
for entry in data:
    subject_types[i] = entry["subj_type"]
    object_types[i] = entry["obj_type"]
    i += 1

In [ ]:
# Extract the unique subject types and object types
unique_subjects = list(Counter(subject_types).keys())
print(f"list of {len(unique_subjects)} subjects {unique_subjects}")

unique_objects = list(Counter(object_types).keys())
print(f"list of {len(unique_objects)} objects {unique_objects}")

## Understanding tools used to obtain the TACRED dataset

How is **text segmentation** done. Let us have a look at the last character in each sentence.

In [ ]:
last_token = []

for entry in data:
    last_token.append(entry["token"][-1])

cnt_last_token = Counter(last_token)

In [ ]:
print(cnt_last_token)

Now we focus on the task of **tokenization** (mainly done by looking at the dataset)

In [ ]:
data[0]["token"]

## Preprocessing
Let us start by creating a list of sentences from the json file, in which data are provided as tokens.

In [3]:
data_list = []
for entry in data:
    data_list.append({"id": entry["id"], "sentence": " ".join(entry["token"])})

dataframe = pd.DataFrame(data_list)
dataframe

,id,sentence
0,61b3a5c8c9a882dcfcd2,Tom Thabane resigned in October last year to f...
1,61b3a65fb9b7111c4ca4,"In 1983 , a year after the rally , Forsberg re..."
2,61b3a65fb9aeb61c81e7,This was among a batch of paperback Oxford Wor...
3,61b3a65fb9c9956eccbc,The latest investigation was authorized after ...
4,61b3a65fb9197aba87ff,The event is a response to a White House immig...
...,...,...
63523,61b3a65fb947cc0bf729,Dodd discussed how he got all his white hair -...
63524,61b3a65fb961852ba56a,"Besides his brother , of Minneapolis , Nolte i..."
63525,61b3a65fb99e781d07cc,Phelps ' win has broken a tie with Mark Spitz ...
63526,61b3a65fb948fa161184,"Susan Neely , president of the American Bevera..."


### Stopword removal

In [4]:
stopwords = set(stopwords.words("english"))


def remove_stopwords(sentence):
    words = sentence.split()  # Tokenize the sentence into words
    filtered_words = [word for word in words if word.lower() not in stopwords]
    return " ".join(filtered_words)


dataframe["sentence_nostop"] = ""
dataframe["sentence_nostop"] = dataframe["sentence"].apply(remove_stopwords)
dataframe.iloc[
    0:5
]  # Now dataframe contains id, tokens and tokens without the stopwords

,id,sentence,sentence_nostop
0,61b3a5c8c9a882dcfcd2,Tom Thabane resigned in October last year to f...,Tom Thabane resigned October last year form Ba...
1,61b3a65fb9b7111c4ca4,"In 1983 , a year after the rally , Forsberg re...","1983 , year rally , Forsberg received so-calle..."
2,61b3a65fb9aeb61c81e7,This was among a batch of paperback Oxford Wor...,among batch paperback Oxford World 's Classics...
3,61b3a65fb9c9956eccbc,The latest investigation was authorized after ...,latest investigation authorized Supreme Court ...
4,61b3a65fb9197aba87ff,The event is a response to a White House immig...,event response White House immigration reform ...


In [5]:
# An example of the missing pronouns
# The subject of this sentence is the pronoun 'he' and the object is 'Arab'.
target_id = "61b3a65fb96dba31ff1d"

# Original sentence
extracted_sentence_with_id = list(dataframe[dataframe["id"] == target_id]["sentence"])

# Modified sentence
extracted_sentence_no_stopword = list(
    dataframe[dataframe["id"] == target_id]["sentence_nostop"]
)

print(f"Original sentence:\n  {extracted_sentence_with_id} ")
print(
    f"Modified sentence:\n  {extracted_sentence_no_stopword}"
)  # the subject is missing due to stopword removal

Original sentence:
  ["But , as a Zionist , he insisted that the city remain under Israeli sovereignty , rejecting Palestinians ' demand to make its Arab part the capital of their would-be state ."] 
Modified sentence:
  [", Zionist , insisted city remain Israeli sovereignty , rejecting Palestinians ' demand make Arab part capital would-be state ."]


The list of stopwords available for the `NLTK` library contains words which could be potential subjects/objects for which we require relation extraction. One example of this are pronouns.\
Additionally also some prepositions which might be useful for detecting relations or in phrasal verbs may be removed.

In the context of the sentence above, the model would be asked to identify the relation between he (SUBJECT) and Arab (OBJECT) as no_relation.\
These examples could not be used by our model if stopwords removal was performed, and this would lead to a smaller dataset which is not ideal, but also to a model that is unable to identify relations including pronouns.


Another issue of removing stopwords is that the provided dataset needs to be re-transformed with all the Stanford Parsing tools. This is because by removing some words the position of other in the senteces is altered.\
Moreover, by reading the TACRED paper, the proposed architecture seems to make use of the position of words in the sentences without stopwords removal. As such, we believe it may not be useful for the task of relation extraction.

### Stemming and Lemmatization process


In [ ]:
def stem_words(sentence):
    stemmer = PorterStemmer()
    words = sentence.split()  # Tokenize the sentence into words
    stemmed_words = [stemmer.stem(word) for word in words]
    return " ".join(stemmed_words)


dataframe["Porter_Stemmer"] = dataframe["sentence"].apply(stem_words)

In [ ]:
# Wordnet lemmatizer does not change the sentences a lot!
def lemmatize_wordnet(sentence):
    lemmatizer = WordNetLemmatizer()
    words = sentence.split()  # Tokenize the sentence into words
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)


dataframe["Wordnet_Lemmatizer"] = dataframe["sentence"].apply(lemmatize_wordnet)

In [ ]:
nlp = stanza.Pipeline(
    "en", tokenize_pretokenized=True, processors="tokenize,lemma"
)  # Only run on small part of data for simplicity


def lemmatize_stanza(sentence):
    doc = nlp(sentence)
    # Extract lemmatized tokens and join them
    lemmatized_tokens = [
        word.lemma for sentence in doc.sentences for word in sentence.words
    ]
    lemmatized_tokens = [word for word in lemmatized_tokens if word is not None]
    if lemmatized_tokens is None:
        return "None"
    else:
        return " ".join(lemmatized_tokens)


dataframe_small = dataframe.iloc[0:100]
dataframe_small["Stanza_Lemmatizer"] = dataframe_small["sentence"].apply(
    lemmatize_stanza
)

In [ ]:
dataframe_small.iloc[0:5]

We believe both stemming and lemmatization to be unnecessary for the task of RE for the following reasons:
- The original dataset already contains information about the grammatical role of each term. As such, altering the individual words would mean that we'd have to recompute their grammatical role, and by doing so we would lose information. 
- By looking at the TACRED paper, it appears that for R.E. tasks the architecture described uses the representations of the words obtained with Stanford NLP tools and not the words themselves. Therefore, preprocessing the data is a waste of computational resources.

## Regex experiments

In [ ]:
# First of all let us extract the relation associated to each sentence and store it in a Pandas dataframe along with additional information which will be needed
data_list = []
for entry in data:
    data_list.append(
        {
            "id": entry["id"],
            "sentence": " ".join(entry["token"]),
            "subj_start": entry["subj_start"],
            "subj_end": entry["subj_end"],
            "obj_start": entry["obj_start"],
            "obj_end": entry["obj_end"],
            "relation": entry["relation"],
        }
    )

# Create a DataFrame
dataframe = pd.DataFrame(data_list)


# Now let us count them
cnt = Counter(dataframe["relation"])
cnt.most_common(10)

In [ ]:
# Extract the unique relations from the 'cnt' dictionary
unique_relations = cnt.keys()
print(list(unique_relations))

In [ ]:
dataframe.iloc[0:5]  # To get the structure of the dataframe

Our idea is to use regular expressions to find and identify the relations. We do not expect this approach to work well but still thought we would try it.\
For simplicity, we will be considering only a subset of the possible relations as each one requires a tailored regex to match it.\
The relations considered are: 
- `org:founded_by`
- `org:member_of`
- `per:spouse`

In all these cases our idea is to use regular expressions where in between the subject and the object of the sentence, there are some terms which point us in the direction of that relation (while also respecting the order of subject and object). We decided to only work with words in between the subject and object as otherwise we would risk classifying each sentence in the same way, regardless of the choice of subject and object.\

Consider the example below:\
Mark, called "The Shark" by his friends, and Lisa just got married.\
Even if I ask the relation between Mark and "The Shark" I would get as a relation married.

In [ ]:
considered_relations = ["no_relation", "org:founded_by", "per:spouse", "org:member_of"]
dataframe_reduced = dataframe.copy()[
    dataframe["relation"].isin(considered_relations)
]  # Only consider part of the dataset with the relations under investigation
dataframe_reduced.reset_index(drop=True, inplace=True)
dataframe_reduced

In [ ]:
# Create a list storing lists of extracted relations
extracted_relations = [[None]] * len(dataframe_reduced)

# Iterate the dataframe
for index, row in dataframe_reduced.iterrows():
    tokens = row["sentence"].split()
    id_ = row["id"]
    subj_start = row["subj_start"]
    subj_end = row["subj_end"]
    obj_start = row["obj_start"]
    obj_end = row["obj_end"]

    # Extract the subject and object of the sentence
    subj = " ".join(tokens[subj_start : subj_end + 1])
    obj = " ".join(tokens[obj_start : obj_end + 1])

    subj = re.escape(subj)
    obj = re.escape(obj)

    # Check for founded_by
    pattern0 = subj + r".*founded by.*" + obj
    pattern1 = (
        obj + r".*(found|founded|founds|founder)(?<!by).*" + subj
    )  # To avoid matching Television founded by Samu (which is matched by the one before)
    if (re.search(pattern0, row["sentence"]) is not None) or (
        re.search(pattern1, row["sentence"]) is not None
    ):
        extracted_relations[index] = extracted_relations[index] + ["org:founded_by"]

    # Check for spouse
    pattern0 = subj + r".*(marry|married|marries).*" + obj
    pattern1 = obj + r".*(marry|married|marries).*" + subj
    pattern2 = subj + r".*" + obj + r"(\w+\s){0,3}" + r"(married|tied the knot)"
    pattern3 = subj + r".*(husband|wife|spouse).*" + obj
    pattern4 = obj + r".*(husband|wife|spouse).*" + subj
    if (
        (re.search(pattern0, row["sentence"]) is not None)
        or (re.search(pattern1, row["sentence"]) is not None)
        or (re.search(pattern2, row["sentence"]) is not None)
        or (re.search(pattern3, row["sentence"]) is not None)
        or (re.search(pattern4, row["sentence"]) is not None)
    ):
        extracted_relations[index] = extracted_relations[index] + ["per:spouse"]

    # Check for member_of
    pattern0 = obj + r".*(member of|members of).*" + subj
    if re.search(pattern0, row["sentence"]) is not None:
        extracted_relations[index] = extracted_relations[index] + ["org:member_of"]

# Now ensure that the extracted relations are in the right format
extracted_relations = [
    entry[1:] if len(entry) > 1 else ["no_relation"] for entry in extracted_relations
]
dataframe_reduced["extracted_relations"] = extracted_relations

Now we assess the performance of our regex attempt.\
We decided to extract the TP and FP for each of the relation types, to compute precision and recall.

In [ ]:
# TP: Number of labels which are correctly classified
TP_Counts = {"per:spouse": 0, "org:member_of": 0, "org:founded_by": 0, "no_relation": 0}

# FP: Number of labels which are wrongly classified
FP_Counts = {"per:spouse": 0, "org:member_of": 0, "org:founded_by": 0, "no_relation": 0}

# True counts of each class
TRUE_Counts = {
    "per:spouse": 0,
    "org:member_of": 0,
    "org:founded_by": 0,
    "no_relation": 0,
}

for index, row in dataframe_reduced.iterrows():
    for rel in TP_Counts.keys():
        if row["relation"] == rel:
            TRUE_Counts[rel] += 1
        if (row["relation"] in row["extracted_relations"]) & (row["relation"] == rel):
            TP_Counts[rel] += 1
        else:
            if (row["relation"] != rel) & (rel in row["extracted_relations"]):
                FP_Counts[rel] += 1

In [ ]:
print(
    f"True Counts:\n  {TRUE_Counts}\nTrue Positives:\n  {TP_Counts}\nFalse Positives:\n  {FP_Counts}\n"
)

In [ ]:
precision_recall = {}

for key in TP_Counts:
    TP = TP_Counts[key]
    FP = FP_Counts[key]
    FN = TRUE_Counts[key] - TP

    precision = TP / (TP + FP) if TP + FP > 0 else 0  # Calculate precision
    recall = TP / (TP + FN) if TP + FN > 0 else 0  # Calculate recall

    precision_recall[key] = {"Precision": precision, "Recall": recall}

# Print precision and recall for each key
for key, values in precision_recall.items():
    print(f'Key: {key}, Precision: {values["Precision"]}, Recall: {values["Recall"]}')